# 02. Build the Komuter page (Version 6)

This notebook draws the second page of the web app, **docs/komuter.html**: KTM Komuter stations by hour. It reads the files made by `07komuter1`, `07komuter2` and `07komuter3`, and follows the same layout as `02buildmap.ipynb`, so the two pages look like sisters.

| Section | What it does |
|---|---|
| 0 | **Visual settings: the place to experiment** (shapes, colours, sizes, lines, text) |
| 1 | Load the prepared files |
| 2 | Base map, boundaries and the Komuter lines |
| 3 | Station markers |
| 4 | Click a station: top destinations, hours, home or work |
| 5 | Title, legend, notes and the hourly slider |
| 6 | Save the page |

## 0. Visual settings

Colours can be any CSS colour: a name (`"red"`, `"teal"`) or a hex code (`"#1f77b4"`).

In [57]:
# ---------------- Station markers ----------------
MARKER_SHAPE     = "circle"    # <- "circle", "square" or "diamond"
MARKER_MIN_SIZE  = 4           # <- width in pixels of the quietest station
MARKER_MAX_SIZE  = 40          # <- width in pixels of the busiest station (KL Sentral)
MARKER_OPACITY   = 1           # <- 0 = invisible, 1 = solid
MARKER_BORDER_COLOUR = "#333333" # <- outline around each marker
MARKER_BORDER_WIDTH  = 1.5       # <- outline thickness in pixels (0 = no outline)
INTERCHANGE_RING_COLOUR = "#333333"  # <- extra ring around stations that connect to Rapid Rail
INTERCHANGE_RING_WIDTH  = 2          # <- ring thickness in pixels (0 = no ring)

# ---------------- Marker colours: home or work ----------------
# home score: 0 = work end (people arrive in the morning), 1 = home end (people leave in the morning)
HOME_COLOURS = ["#e66101", "#f7f7f7", "#0C2363"]   # <- [work end, mixed, home end]
HOME_STOPS   = [0.2, 0.5, 0.8]                     # <- home score at which each colour above is reached
COLOUR_TOO_FEW = "#bbbbbb"                         # <- stations with too few peak trips to judge

# ---------------- Marker colours: weekend ratio (same as the main map) ----------------
COLOUR_COMMUTER = "#E0F2FE"    # <- stations that empty at weekends
COLOUR_MIXED    = "#30BCF8"    # <- stations in between
COLOUR_WEEKEND  = "#0C2363"    # <- stations busier at weekends
RATIO_COMMUTER  = 0.3          # <- ratio at which COLOUR_COMMUTER is reached
RATIO_MIXED     = 0.75         # <- ratio shown in COLOUR_MIXED
RATIO_WEEKEND   = 1.2          # <- ratio at which COLOUR_WEEKEND is reached

# ---------------- Marker colours and track: Komuter lines ----------------
LINE_COLOURS = {                     # <- colour for each line (KTMB's own colours from GTFS)
    "Seremban Line":   "#3C5A9F",
    "Port Klang Line": "#DC2420",
    "Shared core":     "#6E6E6E",    # <- Putra to KL Sentral, used by both lines (station colour only)
}
TRACK_WIDTH   = 4              # <- thickness of the drawn track in pixels
TRACK_OPACITY = 0.45           # <- 0 = invisible, 1 = solid
TRACK_INSERT  = {"Segambut Utara": ("Segambut", "Kepong")}   # <- stations missing from the GTFS timetable, placed between two neighbours

# ---------------- Destination lines (shown when a station is clicked) ----------------
LINE_COLOUR     = "#fbff00"    # <- colour of the lines to the top destinations
LINE_MIN_WIDTH  = 3            # <- thinnest line in pixels
LINE_MAX_WIDTH  = 12           # <- thickest line in pixels (the busiest destination overall)
LINE_OPACITY    = 0.9          # <- 0 = invisible, 1 = solid
LINE_DASHED     = False        # <- True for dashed lines
HIGHLIGHT_COLOUR = "#fbff00"   # <- ring drawn around the clicked station
CHART_ENTRIES_COLOUR = "#0C2363"   # <- bars for entries in the station's 24-hour chart
CHART_EXITS_COLOUR   = "#e66101"   # <- bars for exits in the station's 24-hour chart

# ---------------- Boundaries ----------------
DISTRICT_LINE_COLOUR = "#444444"   # <- district outlines
DISTRICT_LINE_WIDTH  = 2           # <- thickness in pixels
SHOW_DISTRICTS_AT_START = False    # <- True to show district outlines when the page opens

# ---------------- Base map and text ----------------
COLOUR_BY_AT_START = "home"    # <- "home", "ratio" or "line"
BASEMAP     = "OpenStreetMap"  # <- background map; "OpenStreetMap" needs no key
START_ZOOM  = 10               # <- higher = closer in; 10 shows the outer belt too
MAP_CENTRE  = [3.09, 101.64]   # <- [latitude, longitude] the map opens on
FONT        = "Segoe UI, Helvetica, Arial, sans-serif"  # <- font for the title and panels
TITLE       = "MALAYSIA (KLANG VALLEY): KTM KOMUTER BY HOUR"   # <- main heading
SUBTITLE    = ("KTM Komuter trips by station and hour:<br>"
               "&bull; Home end or work end of the daily commute<br>"
               "&bull; The commute hour by hour (slider)<br>"
               "&bull; Where passengers go from each station")
LINK_TEXT   = "&larr; Rapid Rail map"   # <- link back to the main map (docs/index.html)
PANEL_BACKGROUND = "rgba(255, 255, 255, 0.92)"  # <- background of the title, legend and info panels
FT_PREFIX   = ""               # <- shown instead of "W.P. " for Kuala Lumpur and Putrajaya
TOOLTIP_FONT_SIZE = 14         # <- text size in pixels for all hover tooltips
NOTES_OPEN_AT_START = False    # <- True to show the "About the data" panel opened when the page opens

# ---------------- Title panel buttons ----------------
BUTTON_LABELS = {              # <- text on the colour buttons in the title panel
    "home":  "HOME OR WORK",
    "ratio": "WEEKEND RATIO",
    "line":  "LINE",
}

# ---------------- Hourly slider ----------------
HOUR_SPEED_MS   = 700          # <- milliseconds per hour when "Play" is pressed (lower = faster)
HOUR_AT_START   = 24           # <- 0 to 23 = that hour; 24 = the whole day
DAY_AT_START    = "working"    # <- "working" or "weekend_holiday"
DIR_AT_START    = "entries"    # <- "entries" or "exits"
HOUR_NOTE = ("Size follows the chosen hour, day and direction; colours stay as set above.<br>"
             "Try a working day with exits: work ends fill in the morning, home ends in the evening.")   # <- small text under the slider

## 1. Load the prepared files

In [58]:
from pathlib import Path
import json, math, re, zipfile
import pandas as pd
import geopandas as gpd
import folium
from branca.colormap import LinearColormap
from branca.element import MacroElement, Template, Element

PROCESSED = Path("data/processed")
OUTPUT = Path("output")
OUTPUT.mkdir(exist_ok=True)

summary = pd.read_csv(PROCESSED / "komuter_station_summary.csv")
hourly = pd.read_csv(PROCESSED / "komuter_hourly.csv")
flows = pd.read_csv(PROCESSED / "komuter_flows_top.csv")
all_stations = pd.read_csv(PROCESSED / "komuter_stations.csv")      # all stations, including those further away (as trip ends)
districts = gpd.read_file(PROCESSED / "districts.geojson")
kmeta = json.loads((PROCESSED / "komuter_metadata.json").read_text())

summary = summary.sort_values("avg_daily_entries", ascending=False).reset_index(drop=True)
summary["key"] = summary["station"].str.lower().str.replace(r"[^a-z0-9]", "", regex=True)
key_of = dict(zip(all_stations["station"], all_stations["station"].str.lower().str.replace(r"[^a-z0-9]", "", regex=True)))

def show_name(d):
    return str(d).replace("W.P. ", FT_PREFIX)

print(f"{len(summary)} stations, {len(hourly):,} hourly rows, {len(flows)} destination rows; "
      f"period {kmeta['period_start']} to {kmeta['period_end']}")
print("Types:", summary["komuter_type"].value_counts().to_dict())

58 stations, 2,603 hourly rows, 290 destination rows; period 2026-06-01 to 2026-08-31
Types: {'Home-end': 42, 'Work-end': 10, 'Mixed': 4, 'Too few trips': 2}


## 2. Base map, boundaries and the Komuter lines

The track is drawn as straight segments between neighbouring stations, in the order trains call at them in the KTMB GTFS timetable (`data/raw/gtfs_ktmb.zip`). Stations missing from the timetable are placed between two neighbours (`TRACK_INSERT` in Section 0).

In [59]:
# VISUAL: centre, zoom, background
m = folium.Map(location=MAP_CENTRE, zoom_start=START_ZOOM, tiles=BASEMAP, control_scale=True, prefer_canvas=False)
# VISUAL: grey, faded background (remove this line for full colour)
m.get_root().header.add_child(Element("<style>.leaflet-tile-pane { filter: grayscale(70%); opacity: 0.8; }</style>"))
# VISUAL: keeps the layer menu in front of the station panel when it opens
m.get_root().header.add_child(Element("<style>.leaflet-top.leaflet-right { z-index: 1100; }</style>"))
# VISUAL: hover tooltip text size
m.get_root().header.add_child(Element(f"<style>.leaflet-tooltip {{ font-size: {TOOLTIP_FONT_SIZE}px; font-family: {FONT}; line-height: 1.4; }}</style>"))

folium.GeoJson(
    districts, name="Districts", show=SHOW_DISTRICTS_AT_START,
    style_function=lambda f: {"color": DISTRICT_LINE_COLOUR, "weight": DISTRICT_LINE_WIDTH, "fillOpacity": 0},   # VISUAL: district outlines
    tooltip=folium.GeoJsonTooltip(["district"], labels=False),
).add_to(m)

# Station order from the GTFS timetable
GTFS_NAMES = {"BDR TASEK SELATAN": "Bandar Tasek Selatan", "PEL KLANG SEL": "Pelabuhan Klang Selatan",
              "JLN KASTAM": "Jalan Kastam", "KG DATO HARUN": "Kampung Dato Harun", "KG RAJA UDA": "Kampung Raja Uda",
              "PULAU SEBANG/TAMPIN": "Pulau Sebang (Tampin)", "PERHENTIAN MIDVALLEY": "Midvalley",
              "BATU KENTOMENN": "Batu Kentonmen"}
norm = lambda s: re.sub(r"[^a-z0-9]", "", str(s).lower())
by_norm = {norm(s): s for s in all_stations["station"]}
with zipfile.ZipFile("data/raw/gtfs_ktmb.zip") as z:
    routes = pd.read_csv(z.open("routes.txt"))
    trips = pd.read_csv(z.open("trips.txt"), dtype=str)
    times = pd.read_csv(z.open("stop_times.txt"), dtype=str)
    stops = pd.read_csv(z.open("stops.txt"), dtype=str)
stops["station"] = stops["stop_name"].map(lambda n: by_norm.get(norm(GTFS_NAMES.get(n, n))))
komuter_routes = routes[routes["route_short_name"].isin(["Seremban Line", "Port Klang Line"])]
t = times.merge(trips[["trip_id", "route_id"]], on="trip_id").merge(stops[["stop_id", "station"]], on="stop_id")
t = t[t["route_id"].isin(komuter_routes["route_id"])].dropna(subset=["station"])
t["seq"] = t["stop_sequence"].astype(int)
t = t.sort_values(["trip_id", "seq"])
t["next"] = t.groupby("trip_id")["station"].shift(-1)
edges = t.dropna(subset=["next"])
edges = edges[edges["station"] != edges["next"]]
route_name = dict(zip(komuter_routes["route_id"], komuter_routes["route_short_name"]))
edge_set = {}
for r in edges.itertuples():
    pair = tuple(sorted([r.station, r.next]))
    edge_set.setdefault(pair, route_name[r.route_id])
for st, (a, b) in TRACK_INSERT.items():          # e.g. Segambut - Segambut Utara - Kepong
    pair = tuple(sorted([a, b]))
    if pair in edge_set:
        line = edge_set.pop(pair)
        edge_set[tuple(sorted([a, st]))] = line
        edge_set[tuple(sorted([st, b]))] = line

pos = all_stations.set_index("station")[["lat", "lon"]]
track = folium.FeatureGroup(name="Komuter lines").add_to(m)
for (a, b), line in edge_set.items():
    if a in pos.index and b in pos.index:
        folium.PolyLine([pos.loc[a].tolist(), pos.loc[b].tolist()], color=LINE_COLOURS[line],   # VISUAL: track colour per line
                        weight=TRACK_WIDTH, opacity=TRACK_OPACITY, interactive=False).add_to(track)
print(f"Track: {len(edge_set)} segments from the GTFS timetable "
      f"({sum(v == 'Seremban Line' for v in edge_set.values())} Seremban Line, {sum(v == 'Port Klang Line' for v in edge_set.values())} Port Klang Line)")

Track: 58 segments from the GTFS timetable (26 Seremban Line, 32 Port Klang Line)


## 3. Station markers

Marker **area** grows with average daily entries, as on the main map. Each marker carries its three colours (home or work, weekend ratio, line); the buttons in the title panel switch between them. Interchanges with Rapid Rail get an extra ring.

In [60]:
max_entries = summary["avg_daily_entries"].max()

def marker_size(entries):
    return MARKER_MIN_SIZE + (MARKER_MAX_SIZE - MARKER_MIN_SIZE) * math.sqrt(max(entries, 0) / max_entries)

# VISUAL: colours for "Home or work" (HOME_COLOURS and HOME_STOPS in Section 0)
home_scale = LinearColormap(colors=HOME_COLOURS, index=HOME_STOPS, vmin=HOME_STOPS[0], vmax=HOME_STOPS[-1])
# VISUAL: colours for "Weekend ratio"
ratio_scale = LinearColormap(colors=[COLOUR_COMMUTER, COLOUR_MIXED, COLOUR_WEEKEND],
                             index=[RATIO_COMMUTER, RATIO_MIXED, RATIO_WEEKEND], vmin=RATIO_COMMUTER, vmax=RATIO_WEEKEND)

def home_colour(s):
    if s["komuter_type"] == "Too few trips" or pd.isna(s["home_score"]):
        return COLOUR_TOO_FEW
    return home_scale(min(max(s["home_score"], HOME_STOPS[0]), HOME_STOPS[-1]))

def ratio_colour(s):
    return "#999999" if pd.isna(s["weekend_ratio"]) else ratio_scale(min(max(s["weekend_ratio"], RATIO_COMMUTER), RATIO_WEEKEND))

def marker_html(size, colour, attrs="", ring=False):
    shape_css = {"circle": "border-radius: 50%;", "square": "border-radius: 2px;",
                 "diamond": "border-radius: 2px; transform: rotate(45deg) scale(0.8);"}[MARKER_SHAPE]
    ring_css = f"outline:{INTERCHANGE_RING_WIDTH}px solid {INTERCHANGE_RING_COLOUR}; outline-offset:2px; " if ring and INTERCHANGE_RING_WIDTH else ""
    return (f'<div {attrs} style="width:{size:.0f}px; height:{size:.0f}px; background:{colour}; '
            f'opacity:{MARKER_OPACITY}; border:{MARKER_BORDER_WIDTH}px solid {MARKER_BORDER_COLOUR}; {ring_css}'
            f'box-sizing:border-box; box-shadow:0 0 2px rgba(0,0,0,0.5); cursor:pointer; {shape_css}"></div>')

def type_text(s):
    if s["komuter_type"] == "Too few trips":
        return "too few peak trips to judge"
    return f"<b>{s['komuter_type']}</b> (home score {s['home_score']:.2f})"

def tooltip_html(s):
    ic = (f"<span style='color:#555'>Interchange with Rapid Rail: {s['rr_station']}</span><br>"
          if s["interchange"] else "")
    return (f"<b>{s['station']}</b> <span style='color:#777'>({s['corridor']})</span><br>"
            f"{s['avg_daily_entries']:,.0f} entries a day<br>"
            f"Working days: {s['entries_working']:,.0f} &middot; Weekends and holidays: {s['entries_weekend']:,.0f}<br>"
            f"Weekend ratio: {s['weekend_ratio']:.2f}<br>"
            f"Commute: {type_text(s)}<br>" + ic +
            f"<span style='color:#777'>{show_name(s['district'])}</span><br>"
            f"<i style='color:#777'>Click for hours and top destinations</i>")

station_layer = folium.FeatureGroup(name="Stations").add_to(m)
marker_names = {}
for rank, s in summary.iterrows():
    size = marker_size(s["avg_daily_entries"])
    c_home, c_ratio, c_line = home_colour(s), ratio_colour(s), LINE_COLOURS.get(s["corridor"], "#999999")
    colour0 = {"home": c_home, "ratio": c_ratio, "line": c_line}.get(COLOUR_BY_AT_START, c_home)
    mk = folium.Marker(
        location=[s["lat"], s["lon"]],
        icon=folium.DivIcon(html=marker_html(size, colour0, ring=bool(s["interchange"]),
                                             attrs=f'class="stn" data-c-home="{c_home}" data-c-ratio="{c_ratio}" data-c-line="{c_line}"'),
                            icon_size=(size, size), icon_anchor=(size / 2, size / 2)),
        tooltip=folium.Tooltip(tooltip_html(s), sticky=True),
        z_index_offset=rank * 10,          # busier stations first = underneath
    ).add_to(station_layer)
    marker_names[s["key"]] = mk.get_name()
print(f"Drew {len(marker_names)} markers, {MARKER_MIN_SIZE} to {MARKER_MAX_SIZE} px; "
      f"{int(summary['interchange'].sum())} interchanges ringed")

Drew 58 markers, 4 to 40 px; 13 interchanges ringed


## 4. Click a station: top destinations, hours, home or work

Clicking a station draws lines to its top destinations on working days and opens a panel with its commute type and a 24-hour chart of entries and exits. Clicking the map background clears it. **You do not need to edit this cell**: its colours and widths come from Section 0.

In [61]:
DAYS = ["working", "weekend_holiday"]
hour_table = {}
for (st, day), g in hourly.groupby(["station", "day_type"]):
    g = g.set_index("hour")[["entries", "exits"]].reindex(range(24), fill_value=0)
    hour_table.setdefault(key_of[st], {})[day] = {"entries": g["entries"].round(1).tolist(), "exits": g["exits"].round(1).tolist()}
for k in summary["key"]:
    for day in DAYS:
        hour_table.setdefault(k, {}).setdefault(day, {"entries": [0] * 24, "exits": [0] * 24})

station_info = {s["key"]: {
    "name": s["station"], "lat": s["lat"], "lon": s["lon"], "line": s["corridor"], "district": show_name(s["district"]),
    "entries": s["avg_daily_entries"], "size": marker_size(s["avg_daily_entries"]),
    "day": {"working": {"entries": s["entries_working"], "exits": s["exits_working"]},
            "weekend_holiday": {"entries": s["entries_weekend"], "exits": s["exits_weekend"]}},
    "ratio": None if pd.isna(s["weekend_ratio"]) else float(s["weekend_ratio"]),
    "type": s["komuter_type"], "home": None if pd.isna(s["home_score"]) else float(s["home_score"]),
    "colour": home_colour(s), "rr": s["rr_station"] if s["interchange"] else None,
} for _, s in summary.iterrows()}
ends = {key_of[r["station"]]: {"name": r["station"], "lat": r["lat"], "lon": r["lon"]} for _, r in all_stations.iterrows()}

flow_info = {key_of[o]: [[key_of[d], float(v)] for d, v in zip(g["destination"], g["trips_per_day"])]
             for o, g in flows.sort_values("rank").groupby("origin")}
style = {"lineColour": LINE_COLOUR, "lineMin": LINE_MIN_WIDTH, "lineMax": LINE_MAX_WIDTH, "lineOpacity": LINE_OPACITY,
         "dashed": LINE_DASHED, "highlight": HIGHLIGHT_COLOUR, "maxTrips": float(flows["trips_per_day"].max()),
         "topN": int(flows["rank"].max()), "cEntries": CHART_ENTRIES_COLOUR, "cExits": CHART_EXITS_COLOUR,
         "am": kmeta["am_peak"], "pm": kmeta["pm_peak"]}

CLICK_JS = r'''
(function() {
  var map = {{ this._parent.get_name() }};
  var stations = __STATIONS__, ends = __ENDS__, flows = __FLOWS__, hours = __HOURS__, style = __STYLE__;
  var markers = {__MARKERS__};
  var current = null;
  window.kmMap = map; window.kmStations = stations; window.kmMarkers = markers; window.kmHours = hours;
  window.kmRefreshPanel = function() { if (current) { showStation(current); } };
  var layer = L.layerGroup().addTo(map);
  var panel = document.getElementById("flow-panel");
  var fmt = function(n) { return Math.round(n).toLocaleString("en-GB"); };
  var dayName = {working: "working day", weekend_holiday: "weekend or holiday"};

  function clearStation() {
    layer.clearLayers(); current = null;
    panel.innerHTML = "<i>Click a station to see its hours and where its passengers go.</i>";
  }
  function showStation(key) {
    layer.clearLayers(); current = key;
    var o = stations[key], rows = "";
    (flows[key] || []).forEach(function(f, i) {
      var d = ends[f[0]];
      if (!d) { return; }
      var w = style.lineMin + (style.lineMax - style.lineMin) * Math.sqrt(f[1] / style.maxTrips);
      L.polyline([[o.lat, o.lon], [d.lat, d.lon]], {color: style.lineColour, weight: w, opacity: style.lineOpacity,
        dashArray: style.dashed ? "8 6" : null, interactive: false}).addTo(layer);
      rows += "<tr><td>" + (i + 1) + ".</td><td>" + d.name + "</td><td style='text-align:right;padding-left:8px'>" + fmt(f[1]) + "</td></tr>";
    });
    L.circleMarker([o.lat, o.lon], {radius: o.size / 2 + 6, color: style.highlight, weight: 3, fill: false, interactive: false}).addTo(layer);
    var day = window.kmDay || "working";
    panel.innerHTML = "<b>" + o.name + "</b> <span style='color:#777'>(" + o.line + ")</span><br>" +
      "<span style='color:#777'>" + fmt(o.entries) + " entries a day &middot; " + o.district + "</span>" +
      (o.rr ? "<br><span style='color:#555'>Interchange with Rapid Rail: " + o.rr + "</span>" : "") +
      commuteBlock(o) + chart(hours[key][day], day) +
      "<div style='margin-top:10px;border-top:1px solid #ddd;padding-top:6px'><b>Top " + style.topN + " destinations</b> " +
      "<span style='color:#777'>(working days)</span><table style='margin-top:4px;border-collapse:collapse'>" +
      "<tr style='color:#777'><td></td><td>Destination</td><td style='text-align:right;padding-left:8px'>Trips/day</td></tr>" + rows + "</table></div>";
  }
  function commuteBlock(o) {
    var head = "<div style='margin-top:10px;border-top:1px solid #ddd;padding-top:6px'><b>Home or work?</b><br>";
    if (o.home === null || o.type === "Too few trips") {
      return head + "<span style='color:#777'>Too few peak-hour trips to judge.</span></div>";
    }
    var pct = Math.round(o.home * 100);
    var bar = "<div style='position:relative;height:10px;margin:6px 0 2px;border-radius:3px;border:1px solid #ccc;" +
      "background:linear-gradient(to right," + __HOMEGRAD__ + ")'><div style='position:absolute;left:" + pct + "%;top:-4px;" +
      "width:3px;height:16px;margin-left:-1px;background:#000'></div></div>" +
      "<div style='display:flex;justify-content:space-between;font-size:11px;color:#777'><span>work end</span><span>home end</span></div>";
    var words = {"Home-end": "Most people <b>leave</b> from here in the morning and come back in the evening.",
                 "Work-end": "Most people <b>arrive</b> here in the morning and leave in the evening (a workplace, or a change to another line).",
                 "Mixed": "Morning and evening traffic run both ways."}[o.type] || "";
    return head + "<b>" + o.type + "</b> (home score " + o.home.toFixed(2) + ")" + bar +
      "<div style='margin-top:4px;color:#555'>" + words + "</div></div>";
  }
  function chart(h, day) {
    // Entries (left bar) and exits (right bar) in each hour; the dashed line marks the hour chosen on the slider
    var W = 290, H = 90, pad = 4, bw = (W - 2 * pad) / 24;
    var max = Math.max.apply(null, h.entries.concat(h.exits).concat([1]));
    var y = function(v) { return (H - 16) * v / max; };
    var bars = "";
    for (var i = 0; i < 24; i++) {
      var x = pad + i * bw;
      bars += "<rect x='" + (x + 0.5).toFixed(1) + "' y='" + (H - 16 - y(h.entries[i])).toFixed(1) + "' width='" + (bw / 2 - 0.5).toFixed(1) +
              "' height='" + y(h.entries[i]).toFixed(1) + "' fill='" + style.cEntries + "'/>" +
              "<rect x='" + (x + bw / 2).toFixed(1) + "' y='" + (H - 16 - y(h.exits[i])).toFixed(1) + "' width='" + (bw / 2 - 0.5).toFixed(1) +
              "' height='" + y(h.exits[i]).toFixed(1) + "' fill='" + style.cExits + "'/>";
      if (i % 6 === 0) { bars += "<text x='" + x + "' y='" + (H - 3) + "' font-size='10' fill='#777'>" + (i < 10 ? "0" : "") + i + ":00</text>"; }
    }
    var hr = window.kmHour;
    var sel = (hr !== undefined && hr < 24) ? "<line x1='" + (pad + (hr + 0.5) * bw) + "' x2='" + (pad + (hr + 0.5) * bw) +
      "' y1='0' y2='" + (H - 16) + "' stroke='#888' stroke-dasharray='3 2'/>" : "";
    return "<div style='margin-top:10px;border-top:1px solid #ddd;padding-top:6px'><b>By hour</b> <span style='color:#777'>(average " + dayName[day] + ")</span><br>" +
      "<span style='font-size:12px'><span style='color:" + style.cEntries + "'>&#9632;</span> entries &nbsp; " +
      "<span style='color:" + style.cExits + "'>&#9632;</span> exits &nbsp; <span style='color:#777'>(highest " + fmt(max) + " an hour)</span></span>" +
      "<svg width='" + W + "' height='" + H + "' style='display:block;margin-top:4px'>" +
      "<line x1='" + pad + "' x2='" + (W - pad) + "' y1='" + (H - 16) + "' y2='" + (H - 16) + "' stroke='#ccc'/>" + bars + sel + "</svg></div>";
  }
  Object.keys(markers).forEach(function(key) {
    markers[key].on("click", function(e) { L.DomEvent.stopPropagation(e); showStation(key); });
  });
  map.on("click", clearStation);
  clearStation();
})();
'''
home_grad = ",".join(f"{c} {100 * (v - HOME_STOPS[0]) / (HOME_STOPS[-1] - HOME_STOPS[0]):.0f}%" for c, v in zip(HOME_COLOURS, HOME_STOPS))
js = (CLICK_JS.replace("__STATIONS__", json.dumps(station_info)).replace("__ENDS__", json.dumps(ends))
              .replace("__FLOWS__", json.dumps(flow_info)).replace("__HOURS__", json.dumps(hour_table))
              .replace("__STYLE__", json.dumps(style)).replace("__HOMEGRAD__", json.dumps(home_grad))
              .replace("__MARKERS__", ", ".join(f'"{k}": {v}' for k, v in marker_names.items())))
click_js = MacroElement()
click_js._template = Template("{% macro script(this, kwargs) %}" + js + "{% endmacro %}")
m.add_child(click_js)
print(f"Click behaviour added for {len(station_info)} stations ({len(flow_info)} with destinations)")

Click behaviour added for 58 stations (58 with destinations)


## 5. Title, legend, notes and the hourly slider

In [62]:
panel_css = (f"position:absolute; z-index:1000; background:{PANEL_BACKGROUND}; font-family:{FONT}; "   # VISUAL: panel look
             "padding:10px 14px; border-radius:6px; box-shadow:0 1px 5px rgba(0,0,0,0.3); font-size:14px;")
period = f"{pd.Timestamp(kmeta['period_start']):%d %b %Y} to {pd.Timestamp(kmeta['period_end']):%d %b %Y}"
btn = ("border:1px solid #888; background:#fff; border-radius:4px; padding:2px 8px; margin-left:4px; "
       "font-family:inherit; font-size:12px; cursor:pointer;")

switch_html = (f'<div style="margin-top:6px; font-size:13px;"><div>Colour stations by:</div>'
               f'<div style="margin-top:4px; margin-left:-4px;">'
               + "".join(f'<button id="btn-{k}" style="{btn}" onclick="setStationColour(\'{k}\')">{BUTTON_LABELS[k]}</button>' for k in ["home", "ratio", "line"])
               + '</div></div>')
SWITCH_JS = """
<script>
function setStationColour(mode) {
  window.kmColourMode = mode;
  var attr = {home: 'cHome', ratio: 'cRatio', line: 'cLine'}[mode];
  document.querySelectorAll('.stn').forEach(function(el) { el.style.background = el.dataset[attr]; });
  ['home', 'ratio', 'line'].forEach(function(k) {
    document.getElementById('legend-' + k).style.display = (k === mode) ? 'block' : 'none';
    document.getElementById('btn-' + k).style.fontWeight = (k === mode) ? 'bold' : 'normal';
  });
}
window.addEventListener('load', function() { setStationColour('__START__'); });
</script>
""".replace("__START__", COLOUR_BY_AT_START if COLOUR_BY_AT_START in BUTTON_LABELS else "home")

# Title panel, top left
title_html = f"""
<div style="{panel_css} top:14px; left:60px; max-width:600px;">
  <div style="font-size:20px; font-weight:600;">{TITLE}</div>
  <div style="color:#555; margin-top:2px;">{SUBTITLE}</div>
  <div style="color:#777; margin-top:4px; font-size:12px;">Average per day, {period} &middot; <a href="index.html" style="color:#1f5fbf;">{LINK_TEXT}</a></div>
  {switch_html}
</div>"""

# Station panel, top right
flow_panel_html = (f'<div id="flow-panel" style="{panel_css} top:60px; right:12px; width:400px; '
                   f'max-height:calc(100vh - 140px); overflow-y:auto; z-index:1001;"></div>')   # VISUAL: scrolls when long

# Legend, bottom left
examples = [100, 1000, int(round(max_entries, -3))]
size_rows = "".join(
    f'<div style="display:flex; align-items:center; margin:3px 0;">'
    f'<div style="width:{MARKER_MAX_SIZE + 4}px; display:flex; justify-content:center;">{marker_html(marker_size(n), "#bbbbbb")}</div>'
    f'<span style="margin-left:6px;">{n:,} entries a day</span></div>' for n in examples)
ring_row = (f'<div style="display:flex; align-items:center; margin:6px 0 0;"><div style="width:{MARKER_MAX_SIZE + 4}px; display:flex; justify-content:center;">'
            f'{marker_html(12, "#bbbbbb", ring=True)}</div><span style="margin-left:6px;">Interchange with Rapid Rail</span></div>')
home_gradient = f"linear-gradient(to right, {home_grad})"
ratio_gradient = f"linear-gradient(to right, {COLOUR_COMMUTER}, {COLOUR_MIXED}, {COLOUR_WEEKEND})"
line_keys = "".join(
    f'<div style="display:flex; align-items:center; margin:2px 0;"><span style="display:inline-block; width:12px; height:12px; '
    f'background:{c}; border:1px solid {MARKER_BORDER_COLOUR}; margin-right:6px;"></span>{t}</div>' for t, c in LINE_COLOURS.items())
tc = kmeta.get("type_counts", {})
legend_html = f"""
<div style="{panel_css} bottom:50px; left:14px; width:300px;">
  <div style="font-weight:600; margin-bottom:4px;">Station size: level of activity</div>
  {size_rows}{ring_row}
  <div id="legend-home">
  <div style="font-weight:600; margin:8px 0 4px;">Colour: home or work</div>
  <div style="height:12px; border-radius:3px; background:{home_gradient}; border:1px solid #ccc;"></div>
  <div style="display:flex; justify-content:space-between; color:#555; font-size:12px; margin-top:2px;">
    <span>Work end</span><span>Mixed</span><span style="text-align:right">Home end</span>
  </div>
  <div style="display:flex; align-items:center; margin-top:4px; font-size:12px;"><span style="display:inline-block; width:12px; height:12px;
  background:{COLOUR_TOO_FEW}; border:1px solid {MARKER_BORDER_COLOUR}; margin-right:6px;"></span>Too few peak trips to judge</div>
  <div style="color:#777; font-size:12px; margin-top:6px;">Working days: morning entries and evening exits (leaving home, coming back)
  against morning exits and evening entries (arriving at work, leaving). {tc.get('Home-end', 0)} home ends, {tc.get('Work-end', 0)} work ends,
  {tc.get('Mixed', 0)} mixed.</div>
  </div>
  <div id="legend-ratio" style="display:none;">
  <div style="font-weight:600; margin:8px 0 4px;">Colour: weekend ratio</div>
  <div style="height:12px; border-radius:3px; background:{ratio_gradient}; border:1px solid #ccc;"></div>
  <div style="display:flex; justify-content:space-between; color:#555; font-size:12px; margin-top:2px;">
    <span>Commuter<br>{RATIO_COMMUTER}</span><span style="text-align:center">Mixed<br>{RATIO_MIXED}</span><span style="text-align:right">Weekend<br>{RATIO_WEEKEND}+</span>
  </div>
  <div style="color:#777; font-size:12px; margin-top:6px;">Ratio = weekend and public holiday entries divided by working-day entries, as on the Rapid Rail map.</div>
  </div>
  <div id="legend-line" style="display:none;">
  <div style="font-weight:600; margin:8px 0 4px;">Colour: Komuter line</div>
  {line_keys}
  </div>
</div>"""

# Notes, bottom right
notes_html = f"""
<div style="{panel_css} bottom:30px; right:12px; width:300px; font-size:12px; color:#555;">
<details {"open" if NOTES_OPEN_AT_START else ""}>
  <summary style="cursor:pointer;"><b>About the data</b> <span style="color:#999;">(click to open or close)</span></summary>
  KTM Komuter trips by origin, destination and hour, {period} ({kmeta['working_days']} working days, {kmeta['weekend_holiday_days']} weekend and public holiday days). Each trip has one hour, most likely when it started, so exits are shown in the hour the passenger boarded. Trips are kept if at least one end is in the Klang Valley; stations further away appear only as destinations.<br><br>
  Home or work uses the peaks ({kmeta['am_peak']} and {kmeta['pm_peak']}) on working days. Stations where many passengers change to the LRT or MRT can look like work ends: passengers leave Komuter there in the morning.<br><br>
  Track works: the KVDT2 double-track upgrading has changed timetables since 2024; in 2026 the Port Klang to KL Sentral line closes daily from 11:00 to 15:00. Ridership on the affected sections has fallen sharply, so midday hours there reflect the timetable as well as demand.<br><br>
  Track: straight lines between stations in timetable order, not the real alignment.<br><br>
  Sources: data.gov.my (KTMB ridership and GTFS, CC BY 4.0); OpenStreetMap for two station positions; DOSM boundaries; map &copy; OpenStreetMap contributors.<br>
  Last updated: {pd.Timestamp(kmeta['findings_created']):%Y-%m-%d}.
</details>
</div>"""

# Hourly slider, bottom centre
day_sel = "".join(f'<option value="{k}"{" selected" if k == DAY_AT_START else ""}>{v}</option>'
                  for k, v in {"working": "Working day", "weekend_holiday": "Weekend or holiday"}.items())
dir_sel = "".join(f'<option value="{k}"{" selected" if k == DIR_AT_START else ""}>{v}</option>'
                  for k, v in {"entries": "Entries", "exits": "Exits"}.items())
hour_html = f"""
<div id="hourbox" style="{panel_css} bottom:30px; left:50%; transform:translateX(-50%); width:520px;">
  <div style="display:flex; align-items:center; gap:8px;">
    <button id="hr-play" style="{btn} margin-left:0; width:58px;">&#9654; Play</button>
    <input id="hr-slider" type="range" min="0" max="24" value="{HOUR_AT_START}" step="1" style="flex:1; accent-color:#555;">
  </div>
  <div style="display:flex; align-items:center; gap:6px; margin-top:6px;">
    <select id="hr-day" style="{btn} margin-left:0;">{day_sel}</select>
    <select id="hr-dir" style="{btn} margin-left:0;">{dir_sel}</select>
    <b id="hr-label" style="margin-left:4px;"></b>
  </div>
  <div id="hr-net" style="color:#777; font-size:12px; margin-top:2px;"></div>
  <div style="color:#777; font-size:12px;">{HOUR_NOTE}</div>
</div>"""   # VISUAL: slider panel position and width

hour_max = float(hourly[["entries", "exits"]].max().max())
day_max = float(summary[["entries_working", "entries_weekend", "exits_working", "exits_weekend"]].max().max())
HOUR_JS = """
<script>
window.addEventListener('load', function() {
  var cfg = __CFG__;
  var stations = window.kmStations, markers = window.kmMarkers, hours = window.kmHours;
  var slider = document.getElementById('hr-slider'), playBtn = document.getElementById('hr-play');
  var daySel = document.getElementById('hr-day'), dirSel = document.getElementById('hr-dir');
  var timer = null;
  var fmt = function(v) { return Math.round(v).toLocaleString('en-GB'); };
  var size = function(v, max) { return cfg.minSize + (cfg.maxSize - cfg.minSize) * Math.sqrt(Math.max(v, 0) / max); };

  function resize(key, s) {
    var mk = markers[key], icon = mk._icon, o = mk.options.icon.options;
    o.iconSize = [s, s]; o.iconAnchor = [s / 2, s / 2];
    o.html = o.html.replace(/width:[0-9.]+px; height:[0-9.]+px/, 'width:' + s.toFixed(1) + 'px; height:' + s.toFixed(1) + 'px');
    stations[key].size = s;
    if (!icon) { return; }
    icon.style.width = s + 'px'; icon.style.height = s + 'px';
    icon.style.marginLeft = (-s / 2) + 'px'; icon.style.marginTop = (-s / 2) + 'px';
    var box = icon.querySelector('.stn');
    if (box) { box.style.width = s + 'px'; box.style.height = s + 'px'; }
  }
  function show() {
    var h = +slider.value, day = daySel.value, dir = dirSel.value, total = 0;
    window.kmHour = h; window.kmDay = day;
    Object.keys(markers).forEach(function(key) {
      var v = h < 24 ? hours[key][day][dir][h] : stations[key].day[day][dir];
      total += v;
      resize(key, h < 24 ? size(v, cfg.hourMax) : size(v, cfg.dayMax));
    });
    var hh = (h < 10 ? '0' : '') + h;
    document.getElementById('hr-label').textContent = h < 24 ? hh + ':00 to ' + hh + ':59' : 'Whole day';
    document.getElementById('hr-net').textContent = 'All stations: ' + fmt(total) + ' ' + dir + (h < 24 ? ' in this hour' : ' a day') +
      ', average ' + (day === 'working' ? 'working day' : 'weekend or holiday') + (h < 24 ? ' (hour sizes use their own scale)' : '');
    if (window.kmRefreshPanel) { window.kmRefreshPanel(); }
  }
  function stop() { clearInterval(timer); timer = null; playBtn.innerHTML = '&#9654; Play'; }
  playBtn.addEventListener('click', function() {
    if (timer) { stop(); return; }
    if (+slider.value >= 23) { slider.value = cfg.playFrom; show(); }
    playBtn.innerHTML = '&#10074;&#10074; Pause';
    timer = setInterval(function() {
      var i = +slider.value + 1;
      if (i > 23) { stop(); return; }
      slider.value = i; show();
    }, cfg.speed);
  });
  slider.addEventListener('input', function() { stop(); show(); });
  daySel.addEventListener('change', show);
  dirSel.addEventListener('change', show);
  window.kmMap.on('overlayadd', function() { setStationColour(window.kmColourMode || 'home'); show(); });
  show();
});
</script>
""".replace("__CFG__", json.dumps({"minSize": MARKER_MIN_SIZE, "maxSize": MARKER_MAX_SIZE, "hourMax": hour_max, "dayMax": day_max, "speed": HOUR_SPEED_MS, "playFrom": 5}))

for html in (title_html, flow_panel_html, legend_html, notes_html, hour_html, SWITCH_JS, HOUR_JS):
    m.get_root().html.add_child(Element(html))
folium.LayerControl(collapsed=True).add_to(m)

## 6. Save the page

In [63]:
out_file = OUTPUT / "komuter_v6.html"
m.get_root().header.add_child(Element("<title>MALAYSIA (KLANG VALLEY): KTM KOMUTER BY HOUR</title>"))
m.save(out_file)
print(f"Saved {out_file} ({out_file.stat().st_size / 1e3:.0f} KB). Open it in a browser for the full-size view.")

import shutil
Path("docs").mkdir(exist_ok=True)
shutil.copyfile(out_file, Path("docs") / "komuter.html")   # a copy, not a second save
print("Also saved docs/komuter.html, ready to upload to GitHub")
m

Saved output\komuter_v6.html (242 KB). Open it in a browser for the full-size view.
Also saved docs/komuter.html, ready to upload to GitHub


.venv\Scripts\python -m http.server 8000 --directory docs

http://localhost:8000/komuter.html
___
**END**